# 15 - Answering: from retrieved chunks to a cited answer, or a refusal

> **Run order.** Step 15 of the pipeline. Needs 01-06 and notebook 14's `ctx` index.
> See [`notebooks/README.md`](README.md). All logic lives in `src/analyst/`
> (`agent.py`, `llm.py`, `tools.py`, `answer_eval.py`); this notebook orchestrates and shows.

Notebooks 07-14 measured whether the right **chunk** was found. This one measures whether the
right **number** is said, with a citation to it - or whether the system correctly refuses.

The agent keeps the ownership rules from `docs/architecture/overview.md`:

| step | who | what |
|---|---|---|
| route | LLM | intent, company, fiscal years, concept - then **checked against the database** |
| retrieve | Qdrant | dense + query expansion over ADR-008's `ctx` index |
| extract | LLM | *points at* a figure and the evidence block it came from |
| verify | Python | the figure must be **printed** in the cited evidence, or the answer is refused |
| compute | Python | growth rates - the model never does arithmetic |

**No LLM judge.** Every benchmark answer is a number with a known true value, so grading is a
comparison: the true figure at any printed scale (exactly how notebook 06 located it), or a
growth rate within 1 point. Every LLM reply is cached on disk, so re-running this notebook
spends no quota.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

from analyst import answer_eval as ae
from analyst import evaluation as ev
from analyst.agent import K, ask, connect
from analyst.config import get_settings

# The only line to change for another model, e.g. ("groq", "openai/gpt-oss-120b").
PROVIDER, MODEL = "ollama", "llama3.2"

settings = get_settings()
llm, tools = connect(settings, PROVIDER, MODEL)
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
refusals = ae.unanswerable(tools.corpus)
print(f"{llm.name}  k={K}  {len(questions)} answerable + {len(refusals)} unanswerable")
print("filings indexed:", {t: c.filing_years for t, c in tools.corpus.items() if c.filing_years})

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


ollama/llama3.2  k=10  44 answerable + 16 unanswerable

filings indexed:

{'HDFCBANK': (2025,), 'ICICIBANK': (2024, 2025), 'RELIANCE': (2025,), 'SUNPHARMA': (2024, 2025)}

## 1. One question, end to end

A growth question is the hardest path: two separate extractions, each verified, then arithmetic
in Python. The trace is what the API returns - auditable steps, never model monologue.

In [2]:
example = next(q for q in questions if q.question_type == "growth")
a = ask(example.question, llm, tools)

print("Q      :", a.question)
print("ANSWER :", a.answer or f"(refused: {a.abstain_reason})")
print("TRUTH  :", example.answer_display)
for c in a.computations:
    print("COMPUTE:", c.expression, "=", c.result, c.unit)
for c in a.citations:
    print(f"CITES  : {c.ticker} FY{c.fiscal_year} p.{c.pages} {c.snippet[:80]!r}")
print()
steps = [{"step": s.step, "ms": s.ms,
          **{k: v for k, v in s.detail.items() if k not in ("pages", "answer")}} for s in a.trace]
print(pd.DataFrame(steps).to_string(index=False))

Q      :

By what percentage did ICICI Bank's net profit change from FY2024 to FY2025?

ANSWER :

ICICI Bank's net profit changed +23.0% from FY2024 (442,563,735) to FY2025 (544,187,134).

TRUTH  :

...

22.96

percent

CITES  : ICICIBANK FY2024 p.[271] 'CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED CONSOLIDATED BALANCE SHE'

CITES  : ICICIBANK FY2025 p.[266] 'CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED CONSOLIDATED BALANCE SHE'

    step     ms cached intent     tickers     fiscal_years    concept  start  end    k    ticker  fiscal_year       value    unit sources
   route 9483.8  False growth [ICICIBANK] [FY2024, FY2025] Net Income    NaN  NaN  NaN       NaN          NaN         NaN     NaN     NaN
retrieve  195.9    NaN    NaN         NaN              NaN        NaN    NaN  NaN 10.0 ICICIBANK       2024.0         NaN     NaN     NaN
 extract 3541.6   True    NaN         NaN              NaN        NaN    NaN  NaN  NaN       NaN          NaN 442,563,735 million  [6, 9]
retrieve  126.7    NaN    NaN         NaN              NaN        NaN    NaN  NaN 10.0 ICICIBANK       2025.0         NaN     NaN     NaN
 extract 3236.1   True    NaN         NaN              NaN        NaN    NaN  NaN  NaN       NaN          NaN 544,187,134   crore  [8, 9]

## 2. The whole benchmark

44 answerable questions from notebook 06, plus unanswerable ones generated from the corpus: a
company with prices but no indexed report, a future fiscal year, and investment advice.

In [3]:
import time

every = [*questions, *refusals]
t0 = time.perf_counter()
answers = {q.question_id: ask(q.question, llm, tools) for q in every}
grades = [ae.grade(q, answers[q.question_id]) for q in every]
calls = sum(a.llm_calls for a in answers.values())
print(f"{len(every)} questions in {time.perf_counter() - t0:.0f}s, {calls} LLM calls")

run = ae.build_run(llm.name, K, questions, grades, notes="notebook 15")
ae.append_run(run)
ae.write_leaderboard(ae.load_runs())
print(pd.Series(run.metrics).to_string())

60 questions in 295s, 127 LLM calls

n                   44.0000
accuracy             0.2273
value_acc            0.2353
growth_acc           0.2000
wrong_answer         0.5682
cited_expected       0.3182
false_refusal        0.2045
refusal              1.0000
calls_per_q          2.1200
tokens_per_q      2909.0000
p50_ms            4608.2500

## 3. Where it fails

Three outcomes, and they are not equally bad. A **refusal** costs an answer. A **WRONG** answer is
a verified figure from the wrong row or column - the check proves the number is printed in the
evidence, not that it is the number asked for. That is the column to drive down.

In [4]:
truth = {q.question_id: q.answer_display for q in questions}
df = pd.DataFrame([{
    "id": g.question_id, "kind": g.kind,
    "outcome": "correct" if g.correct else ("refused" if g.abstained else "WRONG"),
    "reason": g.reason, "got": ", ".join(answers[g.question_id].values),
    "truth": truth.get(g.question_id, ""), "anchor cited": g.cited_expected,
    "retried": any(s.step == "verify" for s in answers[g.question_id].trace),
} for g in grades])
print(pd.crosstab(df["kind"], df["outcome"]).to_string())
print()
print(df[df["outcome"] != "correct"].to_string(index=False))

outcome       WRONG  correct  refused
kind                                 
growth            5        2        3
unanswerable      0       16        0
value_lookup     20        8        6

                                                  id         kind outcome                reason                        got              truth  anchor cited  retried
                         HDFCBANK-FY2025-TotalAssets value_lookup   WRONG                   NaN              8,363,596,736 Rs 4,818,767 crore         False    False
                   ICICIBANK-FY2024-RetainedEarnings value_lookup   WRONG                   NaN                151,353,548    Rs 89,826 crore          True    False
                          ICICIBANK-FY2025-NetIncome value_lookup   WRONG                   NaN                544,187,134    Rs 51,029 crore          True    False
                       ICICIBANK-FY2025-TotalRevenue value_lookup   WRONG                   NaN                2,947,376.0   Rs 204,715 crore         False     True
                   ICICIBANK-FY2025-RetainedEarnings value_lookup   WRONG                   NaN              3,104,114,654   Rs 118,385 crore          True    False
          

## 4. What the one retry buys

A failed verification triggers one wider look (k=10 -> 20) before refusing. A retry can rescue an
answer - or turn a refusal into a wrong answer, because more evidence is also more rows to
misread.

In [5]:
print(pd.crosstab(df["retried"], df["outcome"], margins=True).to_string())

outcome  WRONG  correct  refused  All
retried                              
False       19       22        0   41
True         6        4        9   19
All         25       26        9   60

## 5. The honest retrieval number: router output vs benchmark labels

Every leaderboard row in notebooks 08-14 filtered and expanded queries with the benchmark's OWN
labels - the ticker, fiscal year and concept stored with each question. A user does not supply
those; the router extracts them. Same retriever, same index, fed the router's reading instead.

In [6]:
from analyst.retrievers import dense, open_store

embedder, store = open_store(settings, "bge-small", "ctx")
labels = dense(embedder, store, "ticker+year", expand=True)


def routed(q, limit):
    r = answers[q.question_id].route
    return labels(q.model_copy(update={
        "ticker": r.tickers[0] if r.tickers else "",
        "fiscal_year": r.fiscal_years[-1] if r.fiscal_years else 0,
        "concept": r.concept or "",
    }), limit)


cfg = ev.RunConfig(retriever="dense+expand[ctx]+router", model="bge-small",
                   filters="ticker+year", limit=max(ev.K_VALUES), points=store.count(),
                   notes=f"ticker/year/concept from {llm.name} routes, not benchmark labels")
routed_run = ev.build_run(cfg, ev.evaluate(questions, routed, max(ev.K_VALUES)), questions,
                          deep=ev.evaluate(questions, routed, max(ev.DEPTHS)), root=ev.ROOT)
ev.append_run(routed_run)
ev.write_leaderboard(ev.load_runs())

curve = {"labels": ev.depth_curve(ev.evaluate(questions, labels, max(ev.DEPTHS))),
         "router": routed_run.depth_curve}
print(pd.DataFrame(curve).T.to_string())
wrong = [(q.question_id, answers[q.question_id].route.model_dump(exclude_none=True))
         for q in questions
         if (answers[q.question_id].route.tickers[:1], answers[q.question_id].route.concept)
         != ([q.ticker], q.concept)]
print(f"\nroutes disagreeing with the labels on ticker or concept: {len(wrong)}")
for w in wrong[:10]:
    print(" ", w)

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


           1       5       10      20      50      100  200
labels  0.1818  0.3182  0.4545  0.6364  0.7955  0.8409  1.0
router  0.1818  0.3864  0.4545  0.6591  0.7727  0.8409  1.0


routes disagreeing with the labels on ticker or concept: 9

('RELIANCE-FY2025-OperatingRevenue', {'intent': 'value_lookup', 'tickers': ['RELIANCE'], 'fiscal_years': [2025], 'concept': 'Total Revenue'})

('RELIANCE-FY2025-StockholdersEquity', {'intent': 'value_lookup', 'tickers': ['RELIANCE'], 'fiscal_years': [2025], 'start': datetime.date(2025, 3, 31), 'end': datetime.date(2025, 3, 31)})

...

('SUNPHARMA-FY2025-OperatingIncome', {'intent': 'value_lookup', 'tickers': ['SUNPHARMA'], 'fiscal_years': [2025]})

('SUNPHARMA-FY2025-OperatingRevenue', {'intent': 'value_lookup', 'tickers': ['SUNPHARMA'], 'fiscal_years': [2025], 'concept': 'Total Revenue'})

('SUNPHARMA-FY2025-StockholdersEquity', {'intent': 'value_lookup', 'tickers': ['SUNPHARMA'], 'fiscal_years': [2025], 'end': datetime.date(2025, 3, 31)})

('SUNPHARMA-FY2024to2025-OperatingRevenue-growth', {'intent': 'growth', 'tickers': ['SUNPHARMA'], 'fiscal_years': [2024, 2025]})

('SUNPHARMA-FY2024to2025-StockholdersEquity-growth', {'intent': 'value_lookup', 'tickers': ['SUNPHARMA'], 'fiscal_years': [2024, 2025]})

## 6. Refusals

In [7]:
print(pd.DataFrame([{
    "question": answers[u.question_id].question, "expected": u.expected,
    "got": answers[u.question_id].abstain_reason or f"ANSWERED: {answers[u.question_id].answer}",
} for u in refusals]).to_string(index=False))

                                                         question             expected                  got
       What was Tata Consultancy Services's net profit in FY2025?        out_of_corpus        out_of_corpus
                         What was Infosys's net profit in FY2025?        out_of_corpus        out_of_corpus
                           What was Wipro's net profit in FY2025?        out_of_corpus        out_of_corpus
                    What was HDFC Bank's total revenue in FY2031?        out_of_corpus        out_of_corpus
                               Should I buy HDFC Bank shares now? unsupported_question unsupported_question
                   What was ICICI Bank's total revenue in FY2031?        out_of_corpus        out_of_corpus
                              Should I buy ICICI Bank shares now? unsupported_question unsupported_question
                   What was Bajaj Finance's net profit in FY2025?        out_of_corpus        out_of_corpus
          What was Reliance 

## 7. The ledger

In [8]:
print(ae.LEADERBOARD.read_text(encoding="utf-8"))

# Answer leaderboard

Generated from `results/answers.jsonl` by `analyst.answer_eval`. Never edit by hand.

> **acc** the true figure at any printed scale; growth within 1 point.
> **wrong** answered but incorrect - the column that matters.
> **cited** a citation names the benchmark's single anchor element, so it is strict.
> **refusal** on generated unanswerable questions.

| run | llm | k | acc | value | growth | wrong | cited | false refusal | refusal | calls/q | tokens/q | p50 s | bench | git |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| llama3.2-k10-2d390e | `ollama/llama3.2` | 10 | 0.227 | 0.235 | 0.200 | 0.568 | 0.318 | 0.204 | 1.000 | 2.1 | 2909 | 4.6 | `2c4aedf3` | `d85d168-dirty` |
